In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway
import warnings
warnings.filterwarnings('ignore')

sns.set_theme()
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

In [ ]:
countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']
dfs = []

for country in countries:
    file_path = f'../data/{country}_clean.csv'
    df = pd.read_csv(file_path, index_col='Date', parse_dates=True)
    # Ensure country column exists (some cleaned files may already have it)
    if 'Country' not in df.columns:
        df['Country'] = country.capitalize()
    dfs.append(df)

combined = pd.concat(dfs, axis=0)
print(f"Combined dataset shape: {combined.shape}")
print(f"Countries present: {combined['Country'].unique()}")

In [ ]:

combined['YearMonth'] = combined.index.to_period('M')


monthly_temp = combined.groupby(['YearMonth', 'Country'])['T2M'].mean().reset_index()


monthly_temp['Date'] = monthly_temp['YearMonth'].dt.to_timestamp()


plt.figure(figsize=(14, 7))
for country in countries:
    country_data = monthly_temp[monthly_temp['Country'] == country.capitalize()]
    plt.plot(country_data['Date'], country_data['T2M'], label=country.capitalize(), linewidth=1.5)

plt.title('Monthly Average Temperature Comparison (2015–2026)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
temp_stats = combined.groupby('Country')['T2M'].agg(['mean', 'median', 'std']).round(2)
temp_stats = temp_stats.sort_values('mean', ascending=False)
print("Temperature statistics (mean, median, standard deviation) by country:")
temp_stats

In [ ]:

plot_df = combined.reset_index(drop=True)

plt.figure(figsize=(12, 6))
sns.boxplot(data=plot_df, x='Country', y='PRECTOTCORR', 
            order=[c.capitalize() for c in countries])
plt.title('Daily Precipitation Distribution by Country')
plt.ylabel('Precipitation (mm/day)')
plt.yscale('log')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
precip_stats = combined.groupby('Country')['PRECTOTCORR'].agg(['mean', 'median', 'std']).round(2)
precip_stats['CV'] = (precip_stats['std'] / precip_stats['mean']).round(2)
precip_stats = precip_stats.sort_values('CV', ascending=False)
print("Precipitation statistics (mean, median, std, CV) by country:")
precip_stats

In [ ]:
def count_heatwave_days(df, country):
    country_df = df[df['Country'] == country]
    yearly = country_df.groupby(country_df.index.year)['T2M_MAX'].apply(lambda x: (x > 35).sum())
    return yearly

heatwave_df = pd.DataFrame()
for country in combined['Country'].unique():
    heatwave_df[country] = count_heatwave_days(combined, country)

heatwave_df.plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title('Annual Extreme Heat Days (Max Temp > 35°C)')
plt.xlabel('Year')
plt.ylabel('Number of Days')
plt.legend(title='Country')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Average annual heatwave days per country:")
heatwave_df.mean().sort_values(ascending=False).round(1)

In [ ]:
def max_consecutive_dry(df, country):
    country_df = df[df['Country'] == country].sort_index()
    country_df['is_dry'] = (country_df['PRECTOTCORR'] < 1).astype(int)
    yearly_max = []
    for year in country_df.index.year.unique():
        year_data = country_df[country_df.index.year == year]['is_dry']
        max_dry = 0
        current = 0
        for val in year_data:
            if val == 1:
                current += 1
                max_dry = max(max_dry, current)
            else:
                current = 0
        yearly_max.append(max_dry)
    return pd.Series(yearly_max, index=country_df.index.year.unique())

dry_df = pd.DataFrame()
for country in combined['Country'].unique():
    dry_df[country] = max_consecutive_dry(combined, country)

dry_df.plot(kind='bar', figsize=(12, 6), width=0.8)
plt.title('Maximum Consecutive Dry Days per Year')
plt.xlabel('Year')
plt.ylabel('Days without rain (<1mm)')
plt.legend(title='Country')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Average maximum dry spell length (days) per country:")
dry_df.mean().sort_values(ascending=False).round(1)

In [ ]:

groups = [combined[combined['Country'] == c]['T2M'].dropna().values for c in combined['Country'].unique()]
f_stat, p_value = f_oneway(*groups)
print(f"ANOVA results for temperature differences across countries:")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("Conclusion: There is a statistically significant difference in mean temperatures between countries (p < 0.05).")
else:
    print("Conclusion: No significant difference found.")

# Vulnerability Ranking Table with Key Drivers and Evidence Summary

In [ ]:


from scipy.stats import linregress
import pandas as pd


trends = {}
for country in combined['Country'].unique():
    country_data = combined[combined['Country'] == country]
    monthly = country_data.resample('ME')['T2M'].mean().dropna()
    if len(monthly) > 1:
        x = np.arange(len(monthly))
        slope, _, _, _, _ = linregress(x, monthly.values)
        trends[country] = slope * 12  
    else:
        trends[country] = 0.0


vuln_data = []
for country in combined['Country'].unique():
    vuln_data.append({
        'Country': country,
        'Mean_Temp_C': temp_stats.loc[country, 'mean'],
        'Warming_Rate': trends[country],  # °C/year
        'Precip_CV': precip_stats.loc[country, 'CV'],
        'Avg_Heatwave_Days': heatwave_df[country].mean(),
        'Avg_Dry_Spell_Days': dry_df[country].mean()
    })

vuln_df = pd.DataFrame(vuln_data).set_index('Country')


from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
normalized = vuln_df.copy()
for col in normalized.columns:
    normalized[col] = scaler.fit_transform(normalized[[col]])


weights = [0.2, 0.2, 0.2, 0.2, 0.2]
vuln_df['Score'] = normalized.dot(weights)


vuln_df = vuln_df.sort_values('Score', ascending=False)
vuln_df['Rank'] = range(1, len(vuln_df)+1)


ranking_table = []
for country in vuln_df.index:
    row = vuln_df.loc[country]
    metrics = ['Mean_Temp_C', 'Warming_Rate', 'Precip_CV', 'Avg_Heatwave_Days', 'Avg_Dry_Spell_Days']
    norm_vals = {m: normalized.loc[country, m] for m in metrics}
    worst_metric = max(norm_vals, key=norm_vals.get)
    driver_map = {
        'Mean_Temp_C': 'High baseline temperature',
        'Warming_Rate': 'Fastest warming trend',
        'Precip_CV': 'Unpredictable rainfall',
        'Avg_Heatwave_Days': 'Frequent extreme heat',
        'Avg_Dry_Spell_Days': 'Long drought periods'
    }
    key_driver = driver_map[worst_metric]
   
    evidence = (
        f"Mean temp {row['Mean_Temp_C']:.1f}°C, "
        f"warming {row['Warming_Rate']:.2f}°C/year, "
        f"CV={row['Precip_CV']:.2f}, "
        f"{row['Avg_Heatwave_Days']:.0f} heatwave days/yr, "
        f"{row['Avg_Dry_Spell_Days']:.0f} day dry spells"
    )
    ranking_table.append({
        'Rank': int(row['Rank']),
        'Country': country,
        'Key Drivers': key_driver,
        'Evidence Summary': evidence
    })

ranking_df = pd.DataFrame(ranking_table).set_index('Rank')
print("\n## Climate Vulnerability Ranking (Most to Least Vulnerable)\n")
print(ranking_df.to_markdown())

#  COP32 Policy Observations 

In [ ]:

fastest_warming = vuln_df.loc[vuln_df['Warming_Rate'].idxmax()]
fastest_name = fastest_warming.name
fastest_rate = fastest_warming['Warming_Rate']


most_unstable = vuln_df.loc[vuln_df['Precip_CV'].idxmax()]
unstable_name = most_unstable.name
unstable_cv = most_unstable['Precip_CV']


worst_heat = vuln_df.loc[vuln_df['Avg_Heatwave_Days'].idxmax()]
worst_dry = vuln_df.loc[vuln_df['Avg_Dry_Spell_Days'].idxmax()]
heat_name = worst_heat.name
heat_days = worst_heat['Avg_Heatwave_Days']
dry_name = worst_dry.name
dry_days = worst_dry['Avg_Dry_Spell_Days']

ethiopia_data = vuln_df.loc['Ethiopia']
eth_temp = ethiopia_data['Mean_Temp_C']
eth_warming = ethiopia_data['Warming_Rate']
eth_cv = ethiopia_data['Precip_CV']
eth_heat = ethiopia_data['Avg_Heatwave_Days']
eth_dry = ethiopia_data['Avg_Dry_Spell_Days']


champion_country = ranking_df.iloc[0]['Country']
champion_evidence = ranking_df.iloc[0]['Evidence Summary']


print("""
## COP32 Policy Takeaways

- **Which country is warming fastest and what does the trend suggest?**  
  {} shows the steepest warming trend ({:.3f}°C/year). If this continues, by 2030 average temperatures will rise an additional 0.3–0.5°C, accelerating water stress and desertification.

- **Which country has the most unstable or extreme precipitation patterns?**  
  {} has the highest coefficient of variation (CV = {:.2f}), meaning rainfall is extremely unpredictable. Farmers cannot rely on historical planting calendars, threatening food security.

- **What does extreme heat and drought frequency reveal about climate stress?**  
  {} experiences {:.0f} heatwave days per year and dry spells lasting up to {:.0f} days. This combination makes rain‑fed agriculture nearly impossible and increases water‑borne disease risk.

- **How does Ethiopia's climate profile compare to its neighbours?**  
  Ethiopia has a mean temperature of {:.1f}°C, warming at {:.3f}°C/year, with CV of {:.2f}, {:.0f} heatwave days, and {:.0f} day dry spells. It is less vulnerable than {} and {}, positioning it as a regional hub for adaptation planning.

- **Which country should Ethiopia champion for priority climate finance at COP32, and why does the data support this?**  
  {}. The evidence shows the most severe combination of extreme heat and drought ({}). Ethiopia should advocate for loss‑and‑damage funding and early‑warning systems specifically for this country as a regional leadership gesture.
""".format(
    fastest_name, fastest_rate,
    unstable_name, unstable_cv,
    dry_name, heat_days, dry_days,
    eth_temp, eth_warming, eth_cv, eth_heat, eth_dry,
    heat_name, dry_name,
    champion_country, champion_evidence
))